# 膝关节间隙宽度（JSN）测量 - 实验 Notebook

本 Notebook 演示膝关节间隙测量流水线，共 5 步：

1. **加载图像与 mask** - 加载原始图像与分割 mask，显示基本信息
2. **提取骨缘** - 用多种方法获取股骨下缘与胫骨上缘
3. **内外侧划分** - 可视化内外侧区室（分割线 + 着色点）
4. **计算 JSN** - 按区室计算关节间隙宽度并可视化最小距离
5. **批量测试** - 对多例运行流水线并汇总结果

## 步骤 0：配置与导入

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import SimpleITK as sitk
from scipy.spatial.distance import cdist

# KOA package root (parent of notebooks/ when cwd is notebooks)
_nb = Path.cwd().resolve()
KNEE_PKG_ROOT = _nb.parent if _nb.name == "notebooks" else _nb
if str(KNEE_PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(KNEE_PKG_ROOT))

from koa.configs.jsn_config import (
    CURRENT_CONFIG_KEY,
    JOINT_SPACE_MEASUREMENT_CONFIGS,
)
from koa.utils.sitk_utils import load_sitk_image
from koa.jwd import direction, edges, jsn, compartments

# 读取配置
config = dict(JOINT_SPACE_MEASUREMENT_CONFIGS[CURRENT_CONFIG_KEY])
label_mapping = config["label_mapping"]

print("当前配置 key:", CURRENT_CONFIG_KEY)
print("标签映射:", label_mapping)
print("边缘方法:", config.get("edge_method"))
print("距离方法:", config.get("distance_method"))
print("JSN 狭窄阈值:", config.get("jsn_narrow_mm"), "mm")

## Notebook 参数

结果 CSV 目录默认与 **`jsn_config`** 的 **`output_csv`** 同目录；批量出图默认写入 **`output_figure_dir`**（与 CSV **同级的 `figures` 目录**）。出图格式与 **`osteophyte.ipynb`** 对齐：使用 **`FIG_EXT`**（如 `"png"` / `"pdf"`）与 **`FIG_DPI`**。评估用标签表、指标输出仍默认在 CSV 所在目录（`EVAL_*` 文件名）。改参数后请先 **运行本段下一格代码**再继续后面步骤。

In [ ]:
# =============================================================================
# Notebook parameters — edit here; re-run this cell after changes.
# =============================================================================

# --- Step 1: single-case demo ---
CASE_INDEX = 0  # index into ``case_ids`` from Step 1
CASE_ID_OVERRIDE = ""  # non-empty stem overrides CASE_INDEX (e.g. exact mask filename without .nrrd)

# --- Step 5: batch pipeline outputs (defaults from ``config["output_csv"]`` → ``jsn_config``) ---
_cfg_jsn_csv = Path(config["output_csv"]).expanduser().resolve()
JSN_OUTPUT_DIR = _cfg_jsn_csv.parent  # same directory as pipeline CSV
JSN_RESULTS_CSV_NAME = _cfg_jsn_csv.name
# To use another folder/file name, assign ``JSN_OUTPUT_DIR`` / ``JSN_RESULTS_CSV_NAME`` after the lines above.
SAVE_JSN_RESULTS_CSV = True  # True -> write ``JSN_OUTPUT_DIR / JSN_RESULTS_CSV_NAME`` (equals ``config['output_csv']`` when defaults kept)

# --- Step 5: batch presentation figures (``config["output_figure_dir"]``; jsn_config default = CSV dir / ``figures``) ---
BATCH_EXPORT_FIGURES = True
_raw_fig = config.get("output_figure_dir")
JSN_OUTPUT_FIGURE_DIR = (
    Path(_raw_fig).expanduser().resolve()
    if _raw_fig
    else (_cfg_jsn_csv.parent / "figures").resolve()
)
# Override: ``JSN_OUTPUT_FIGURE_DIR = Path("...")``
BATCH_MAX_CASES = 100
BATCH_FIGSIZE = (10, 10)
# Same naming as ``osteophyte.ipynb``: single file type per figure.
FIG_EXT = "png"  # e.g. "png", "pdf", "svg"
FIG_DPI = 150  # passed to ``savefig`` (see osteophyte notebook)

# --- Step 4b presentation crop (also used in batch export) ---
PRESENTATION_FIGSIZE = (10, 10)
JSN_ZOOM_MARGIN_FRAC = 0.58
JSN_ZOOM_MIN_PAD_PX = 120

# --- Evaluation section (label CSV / metrics output) ---
EVAL_CSV_NAME = "jwd_result_w_label.csv"
EVAL_XLSX_NAME = "jsn_results_w_labels.xlsx"
EVAL_METRICS_CSV_NAME = "jsn_evaluation_metrics.csv"
EVAL_IGNORE_CASE_IDS = frozenset({
    "KOA_卓大均10880012_074Y_F",
    "KOA_周显智10933847_055Y_M",
    "KOA_赵秀兰10689968_070Y_F",
})

In [ ]:
# ============ 可视化辅助函数 ============

def mask_to_rgb(mask_2d, label_mapping):
    """
    Convert multi-label mask to RGB image.
    Colors follow 3D Slicer order: 1=green, 2=yellow, 3=red, 4=blue
    """
    H, W = mask_2d.shape
    rgb = np.zeros((H, W, 3), dtype=np.float32)
    colors = {
        "background": [0, 0, 0],
        "Femur_R": [0, 1, 0],      # 1 green
        "Tibia_R": [1, 1, 0],      # 2 yellow
        "Femur_L": [1, 0, 0],      # 3 red
        "Tibia_L": [0, 0, 1],      # 4 blue
    }
    for label_name, label_val in label_mapping.items():
        if label_name in colors:
            mask_region = mask_2d == label_val
            rgb[mask_region] = colors[label_name]
    return rgb


def plot_image_and_mask(image_2d, mask_2d, label_mapping, title="", alpha=0.5):
    """
    Display original image with colored mask overlaid on top.
    Mask colors (3D Slicer): 1=green, 2=yellow, 3=red, 4=blue.
    """
    mask_rgb = mask_to_rgb(mask_2d, label_mapping)
    # Normalize image to [0,1] for consistent display
    img = image_2d.astype(np.float64)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    img_rgb = np.stack([img, img, img], axis=-1)
    
    # Overlay: where mask is non-background, blend mask color with image
    has_mask = (mask_2d > 0)
    overlay = img_rgb.copy()
    overlay[has_mask] = (1 - alpha) * img_rgb[has_mask] + alpha * mask_rgb[has_mask]
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(overlay)
    ax.set_title("Image + Mask (1=green, 2=yellow, 3=red, 4=blue, 3D Slicer)")
    ax.axis("off")
    if title:
        fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()
    return mask_rgb


def make_image_mask_overlay(image_2d, mask_2d, label_mapping, alpha=0.5):
    """
    Build image + mask overlay (same as Step 1). Returns RGB array for use as background.
    Edge extraction (Step 2) is mask-based; drawing on this overlay shows bone + mask.
    """
    mask_rgb = mask_to_rgb(mask_2d, label_mapping)
    img = np.asarray(image_2d, dtype=np.float64)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    img_rgb = np.stack([img, img, img], axis=-1)
    has_mask = (np.asarray(mask_2d, dtype=np.int64) > 0)
    overlay = img_rgb.copy()
    overlay[has_mask] = (1 - alpha) * img_rgb[has_mask] + alpha * mask_rgb[has_mask]
    return overlay


def grayscale_display_rgb(image_2d):
    """仅归一化 X 光 → (H,W,3) 灰度 RGB，用于展示图（不叠分割 mask）。"""
    img = np.asarray(image_2d, dtype=np.float64)
    lo, hi = np.nanmin(img), np.nanmax(img)
    g = (img - lo) / (hi - lo + 1e-8)
    g = np.clip(g, 0.0, 1.0)
    return np.stack([g, g, g], axis=-1).astype(np.float32)


def zoom_limits_from_jsn(jsn_result, image_shape, *, margin_frac=None, min_pad_px=None):
    """
    Crop around all compartment min-distance pairs with **generous** padding so the **whole knee**
    (distal femur + proximal tibia region) stays visible — closer to clinical reference layouts.
    Defaults follow notebook **参数** cell: ``JSN_ZOOM_MARGIN_FRAC``, ``JSN_ZOOM_MIN_PAD_PX``.
    Returns ``(c0, c1, r0, r1)``; ``None`` if no valid points.
    """
    try:
        _mf0, _mp0 = JSN_ZOOM_MARGIN_FRAC, JSN_ZOOM_MIN_PAD_PX
    except NameError:
        _mf0, _mp0 = 0.58, 120
    mf = _mf0 if margin_frac is None else float(margin_frac)
    mp = _mp0 if min_pad_px is None else int(min_pad_px)
    rows, cols = [], []
    for side in ("left", "right"):
        for part in ("medial", "lateral"):
            fp = jsn_result.get(f"jsn_{side}_{part}_femur_pt")
            tp = jsn_result.get(f"jsn_{side}_{part}_tibia_pt")
            if fp is None or tp is None:
                continue
            rows.extend([float(fp[0]), float(tp[0])])
            cols.extend([float(fp[1]), float(tp[1])])
    if not rows:
        return None
    H, W = int(image_shape[0]), int(image_shape[1])
    r0, r1 = min(rows), max(rows)
    c0, c1 = min(cols), max(cols)
    rh, cw = max(1e-6, r1 - r0), max(1e-6, c1 - c0)
    # Wide context: fraction of joint bbox + fraction of full image (shows more knee than tight crop)
    pad_r = max(float(mp), rh * mf, rh * 0.62, H * 0.17)
    pad_c = max(float(mp), cw * mf, cw * 0.62, W * 0.15)
    r0i = int(max(0, np.floor(r0 - pad_r)))
    r1i = int(min(H - 1, np.ceil(r1 + pad_r)))
    c0i = int(max(0, np.floor(c0 - pad_c)))
    c1i = int(min(W - 1, np.ceil(c1 + pad_c)))
    if r1i <= r0i or c1i <= c0i:
        return None
    return (c0i, c1i, r0i, r1i)


def apply_axis_zoom(ax, lims):
    """``lims`` 为 ``(c0, c1, r0, r1)``；``imshow`` 默认 y 向下增大。"""
    if lims is None:
        return
    c0, c1, r0, r1 = lims
    ax.set_xlim(c0, c1)
    ax.set_ylim(r1, r0)


def plot_edges_on_mask(background, femur_pts, tibia_pts, title="", ax=None):
    """
    Plot femur lower edge and tibia upper edge on background.
    Points are (row, col); scatter uses (col, row) = (x, y).
    """
    show_now = ax is None
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(8, 10))
    ax.imshow(background)
    if len(femur_pts) > 0:
        ax.scatter(femur_pts[:, 1], femur_pts[:, 0], c="white", s=10, edgecolors="black", linewidths=0.5, label=f"femur ({len(femur_pts)})")
    if len(tibia_pts) > 0:
        ax.scatter(tibia_pts[:, 1], tibia_pts[:, 0], c="cyan", s=10, edgecolors="black", linewidths=0.5, label=f"tibia ({len(tibia_pts)})")
    ax.legend(loc="upper right", fontsize=8)
    ax.set_title(title)
    ax.axis("off")
    if show_now:
        plt.tight_layout()
        plt.show()


def plot_edges_on_mask_by_side(background, left_femur, left_tibia, right_femur, right_tibia, title="", ax=None):
    """
    Plot femur/tibia edges with colors matching mask (3D Slicer):
    right femur=green, right tibia=yellow, left femur=red, left tibia=blue.
    Points are (row, col); scatter uses (col, row).
    """
    show_now = ax is None
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(8, 10))
    ax.imshow(background)
    if len(right_femur) > 0:
        ax.scatter(right_femur[:, 1], right_femur[:, 0], c="lime", s=8, edgecolors="darkgreen", linewidths=0.3, label=f"R femur ({len(right_femur)})")
    if len(right_tibia) > 0:
        ax.scatter(right_tibia[:, 1], right_tibia[:, 0], c="yellow", s=8, edgecolors="orange", linewidths=0.3, label=f"R tibia ({len(right_tibia)})")
    if len(left_femur) > 0:
        ax.scatter(left_femur[:, 1], left_femur[:, 0], c="red", s=8, edgecolors="darkred", linewidths=0.3, label=f"L femur ({len(left_femur)})")
    if len(left_tibia) > 0:
        ax.scatter(left_tibia[:, 1], left_tibia[:, 0], c="blue", s=8, edgecolors="navy", linewidths=0.3, label=f"L tibia ({len(left_tibia)})")
    ax.legend(loc="upper right", fontsize=7)
    ax.set_title(title)
    ax.axis("off")
    if show_now:
        plt.tight_layout()
        plt.show()


def draw_min_distance_pair(ax, f_pt, t_pt, jsn_mm, image_shape, fontsize=10, offset_frac=0.08,
                           compartment_label=None, color="yellow", draw_labels=True,
                           *, presentation=False, is_narrow=False, presentation_n_stripes=4):
    """
    Draw min-distance visualization (femur / tibia closest pair).

    - ``presentation=False`` (默认): 虚线 + 小点，常与骨缘散点同图（调试）。
    - ``presentation=True``: **参考临床示意图** — 股骨–胫骨最近点对之间画 ``presentation_n_stripes``
      条**竖直线**（品红色等由调用方传入），**不画大端点**；``is_narrow=True`` 时加粗竖线。

    f_pt/t_pt are (row, col). compartment_label e.g. "L-med".
    """
    leg = f"{compartment_label or 'JSN'} {jsn_mm:.2f}mm" + (" [NARROW]" if is_narrow else "")
    if presentation:
        lw = 3.2 if is_narrow else 2.2
        fr, fc = float(f_pt[0]), float(f_pt[1])
        tr, tc = float(t_pt[0]), float(t_pt[1])
        r_lo, r_hi = min(fr, tr), max(fr, tr)
        n = max(2, int(presentation_n_stripes))
        if abs(fc - tc) < 1e-6:
            xs = np.full(n, fc, dtype=np.float64)
        else:
            xs = np.linspace(fc, tc, n)
        for i, x in enumerate(xs):
            ax.plot(
                [x, x], [r_lo, r_hi], "-",
                color=color, lw=lw, solid_capstyle="butt", zorder=6,
                label=(leg if i == 0 else "_nolegend_"),
            )
    else:
        lw = 2.8 if is_narrow else 2.0
        ax.plot([f_pt[1], t_pt[1]], [f_pt[0], t_pt[0]], "--", color=color, lw=lw,
                label=leg)
        ax.scatter([f_pt[1], t_pt[1]], [f_pt[0], t_pt[0]], c=color, s=22 if is_narrow else 18,
                   marker="o", zorder=5, edgecolors="white", linewidths=1)
    if not draw_labels:
        return
    offset = min(image_shape[0], image_shape[1]) * offset_frac
    lbl = f"Femur ({compartment_label})" if compartment_label else "Femur (min)"
    ax.annotate(lbl, xy=(f_pt[1], f_pt[0]), xytext=(f_pt[1] - offset, f_pt[0] - offset),
                arrowprops=dict(arrowstyle="->", color=color, lw=2), fontsize=fontsize, color="white",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="black", alpha=0.9))
    lbl = f"Tibia ({compartment_label})" if compartment_label else "Tibia (min)"
    ax.annotate(lbl, xy=(t_pt[1], t_pt[0]), xytext=(t_pt[1] + offset, t_pt[0] + offset),
                arrowprops=dict(arrowstyle="->", color=color, lw=2), fontsize=fontsize, color="white",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="black", alpha=0.9))


def plot_jsn_with_line(mask_rgb, fem_pts, tib_pts, spacing, title="", ax=None):
    """
    Plot edges and draw the minimum distance line with JSN value.
    使用 jsn.get_min_distance_pair 得到最短距离点对，保证与结果一致。
    """
    show_now = ax is None
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(8, 10))
    
    ax.imshow(mask_rgb)
    
    if len(fem_pts) > 0 and len(tib_pts) > 0:
        # 使用 jsn.get_min_distance_pair，与 pipeline 结果一致
        jsn_mm, status, f_pt, t_pt = jsn.get_min_distance_pair(fem_pts, tib_pts, spacing)
        ax.scatter(fem_pts[:, 1], fem_pts[:, 0], c="white", s=8, edgecolors="black", linewidths=0.3)
        ax.scatter(tib_pts[:, 1], tib_pts[:, 0], c="cyan", s=8, edgecolors="black", linewidths=0.3)
        if status == "ok" and f_pt is not None and t_pt is not None:
            draw_min_distance_pair(ax, f_pt, t_pt, jsn_mm, mask_rgb.shape, fontsize=11, offset_frac=0.08)
        ax.legend(loc="upper right", fontsize=10)
    
    ax.set_title(title)
    ax.axis("off")
    
    if show_now:
        plt.tight_layout()
        plt.show()


def plot_medial_lateral_split(mask_rgb, mid_col, med_fem, med_tib, lat_fem, lat_tib, 
                               side, spacing, title="", ax=None, compute_jsn=True):
    """
    Plot medial/lateral split with vertical dividing line.
    """
    show_now = ax is None
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(8, 10))
    
    H = mask_rgb.shape[0]
    ax.imshow(mask_rgb)
    
    # Draw vertical split line
    ax.axvline(x=mid_col, color="magenta", linestyle="--", lw=2, label="split line")
    
    # Plot medial points (one color) and lateral points (another color)
    if len(med_fem) > 0:
        ax.scatter(med_fem[:, 1], med_fem[:, 0], c="white", s=12, marker="o", label="medial femur")
    if len(med_tib) > 0:
        ax.scatter(med_tib[:, 1], med_tib[:, 0], c="cyan", s=12, marker="o", label="medial tibia")
    if len(lat_fem) > 0:
        ax.scatter(lat_fem[:, 1], lat_fem[:, 0], c="orange", s=12, marker="s", label="lateral femur")
    if len(lat_tib) > 0:
        ax.scatter(lat_tib[:, 1], lat_tib[:, 0], c="lime", s=12, marker="s", label="lateral tibia")
    
    jsn_med = jsn_lat = float("nan")
    if compute_jsn:
        # Compute JSN for each compartment (for quick preview only)
        sp = (float(spacing[0]), float(spacing[1]))
        if len(med_fem) > 0 and len(med_tib) > 0:
            d = cdist(med_fem * sp, med_tib * sp)
            jsn_med = float(np.min(d))
        if len(lat_fem) > 0 and len(lat_tib) > 0:
            d = cdist(lat_fem * sp, lat_tib * sp)
            jsn_lat = float(np.min(d))
    
    ax.legend(loc="upper right", fontsize=7)
    if compute_jsn:
        ax.set_title(f"{title}\nMedial JSN={jsn_med:.2f}mm, Lateral JSN={jsn_lat:.2f}mm")
    else:
        ax.set_title(title)
    ax.axis("off")
    
    if show_now:
        plt.tight_layout()
        plt.show()
    
    return jsn_med, jsn_lat


print("可视化辅助函数已定义。")

## Step 1: 加载图像与 Mask

加载一例的原始 X 光图像与分割 mask。

In [ ]:
# List available cases
mask_dir = Path(config["mask_dir"])
image_dir = Path(config["image_dir"])

# Find mask files (exclude *_0000.nrrd which are original images)
mask_files = sorted([f for f in mask_dir.glob("*.nrrd")])
print(f"在 {mask_dir} 中找到 {len(mask_files)} 个 mask 文件")

# Show first few case IDs
case_ids = [f.stem for f in mask_files]
print("示例 case ID:", case_ids)

In [ ]:
# Select a case to analyze (see **Notebook 参数**)
if CASE_ID_OVERRIDE.strip():
    CASE_ID = CASE_ID_OVERRIDE.strip()
elif case_ids:
    _idx = int(np.clip(CASE_INDEX, 0, len(case_ids) - 1))
    CASE_ID = case_ids[_idx]
else:
    CASE_ID = "sample_case"
print(f"当前选中 case: {CASE_ID} (CASE_INDEX={CASE_INDEX}, override={CASE_ID_OVERRIDE!r})")

# Load image and mask
image_path = image_dir / f"{CASE_ID}_0000.nrrd"
mask_path = mask_dir / f"{CASE_ID}.nrrd"

print(f"图像路径: {image_path}")
print(f"Mask 路径: {mask_path}")
print(f"图像存在: {image_path.exists()}")
print(f"Mask 存在: {mask_path.exists()}")

In [ ]:
# Load and display
image_sitk = load_sitk_image(image_path)
mask_sitk = load_sitk_image(mask_path)

image_arr = sitk.GetArrayFromImage(image_sitk)
mask_arr = sitk.GetArrayFromImage(mask_sitk)

# Handle 3D arrays (take first slice)
image_2d = image_arr[0] if image_arr.ndim == 3 else image_arr
mask_2d = mask_arr[0] if mask_arr.ndim == 3 else mask_arr

spacing = mask_sitk.GetSpacing()

print("\n=== 图像信息 ===")
print(f"  Shape: {image_2d.shape}")
print(f"  Spacing: {spacing}")
print(f"  数值范围: [{image_2d.min()}, {image_2d.max()}]")

print("\n=== Mask 信息 ===")
print(f"  Shape: {mask_2d.shape}")
print(f"  唯一取值: {np.unique(mask_2d)}")
for name, val in label_mapping.items():
    count = np.sum(mask_2d == val)
    if count > 0:
        print(f"  {name} (val={val}): {count} pixels")

In [ ]:
# Display image and mask side by side
mask_rgb = plot_image_and_mask(image_2d, mask_2d, label_mapping, title=f"Case: {CASE_ID}", alpha=0.4)

## Step 2: 提取股骨下缘与胫骨上缘

比较 3 种方法：
- **distance_percentile**：保留到对侧骨距离最小的轮廓点
- **axis_projection**：沿垂直轴取极值行
- **morphological**：膨胀 mask 后取边缘带

In [ ]:
# First, infer axis direction
axis_info = direction.infer_axis(config, mask_2d, spacing, mask_sitk_or_path=mask_sitk)
print("轴向信息:")
print(f"  axis_si (0=row, 1=col): {axis_info['axis_si']}")
print(f"  head_is_low (头端在较小索引): {axis_info['head_is_low']}")

In [ ]:
# Extract edges using all 3 methods and compare
methods = ["distance_percentile", "axis_projection", "morphological"]
results_by_method = {}

for method in methods:
    cfg = {**config, "edge_method": method}
    edges_result = edges.extract_femur_tibia_edges(mask_2d, spacing, label_mapping, axis_info, cfg)
    results_by_method[method] = edges_result
    
    print(f"\n=== 方法: {method} ===")
    for side in ["left", "right"]:
        if side in edges_result:
            fem_pts, tib_pts = edges_result[side]
            print(f"  {side}: 股骨 {len(fem_pts)} 点, 胫骨 {len(tib_pts)} 点")

In [ ]:
# Visualize all 3 methods: draw BOTH left and right femur/tibia edges on each subplot.
overlay_step2 = make_image_mask_overlay(image_2d, mask_2d, label_mapping, alpha=0.5)
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

for idx, method in enumerate(methods):
    ax = axes[idx]
    edges_result = results_by_method[method]
    left_fem, left_tib = np.zeros((0, 2)), np.zeros((0, 2))
    right_fem, right_tib = np.zeros((0, 2)), np.zeros((0, 2))
    for side in ["left", "right"]:
        if side not in edges_result:
            continue
        f, t = edges_result[side]
        if side == "left":
            left_fem = f
            left_tib = t
        else:
            right_fem = f
            right_tib = t
    if len(left_fem) > 0 or len(left_tib) > 0 or len(right_fem) > 0 or len(right_tib) > 0:
        plot_edges_on_mask_by_side(overlay_step2, left_fem, left_tib, right_fem, right_tib, title=f"{method}\n(left + right)", ax=ax)
    else:
        ax.imshow(overlay_step2)
        ax.set_title(f"{method}\nNo edge data")
        ax.axis("off")

plt.suptitle(f"Step 2: Edge Extraction (image+mask; femur lower / tibia upper, both sides)", fontsize=14)
plt.tight_layout()
plt.show()

## Step 3: 内外侧划分（可视化）

可视化每侧膝关节如何按中线划分为内侧与外侧区室（分割线 + 点）。

In [ ]:
# Select edge extraction method (from Step 2 results)
selected_method = config.get("edge_method", "morphological")
edges_per_side = results_by_method[selected_method]  # {side: (fem_pts, tib_pts)}

# Split each side into medial/lateral compartments
edges_result = compartments.split_medial_lateral(
    edges_per_side,
    mask_2d.shape[1],
    spacing=tuple(spacing) if spacing else None,
    exclude_near_midline_ratio=config.get("exclude_near_midline_ratio"),
    exclude_near_midline_mm=config.get("exclude_near_midline_mm"),
)

print(f"按中线缩短的mm数: {config.get('exclude_near_midline_mm')}")
print(f"按中线缩短的比例: {config.get('exclude_near_midline_ratio')}")


print(f"使用边缘方法: {selected_method}")
print("Step 3: 内外侧划分可视化（此处不计算 JSN）。")

In [ ]:
# Step 3: 原图+mask+edge点+中线，左右画在一起；用颜色区分内侧(medial)与外侧(lateral)
overlay_step3 = make_image_mask_overlay(image_2d, mask_2d, label_mapping, alpha=0.5)
fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.imshow(overlay_step3)

# 按内侧/外侧聚合点（左右两侧都收进来）
med_fem_all = np.zeros((0, 2))
med_tib_all = np.zeros((0, 2))
lat_fem_all = np.zeros((0, 2))
lat_tib_all = np.zeros((0, 2))
mid_cols = {}

for side in ["left", "right"]:
    if side not in edges_result:
        mid_cols[side] = mask_2d.shape[1] / 2
        continue

    fem_med, tib_med = edges_result[side]["medial"]
    fem_lat, tib_lat = edges_result[side]["lateral"]

    if len(fem_med) > 0:
        med_fem_all = np.vstack([med_fem_all, fem_med]) if len(med_fem_all) > 0 else fem_med
    if len(tib_med) > 0:
        med_tib_all = np.vstack([med_tib_all, tib_med]) if len(med_tib_all) > 0 else tib_med
    if len(fem_lat) > 0:
        lat_fem_all = np.vstack([lat_fem_all, fem_lat]) if len(lat_fem_all) > 0 else fem_lat
    if len(tib_lat) > 0:
        lat_tib_all = np.vstack([lat_tib_all, tib_lat]) if len(lat_tib_all) > 0 else tib_lat

    all_pts = [fem_med, tib_med, fem_lat, tib_lat]
    combined = np.vstack([p for p in all_pts if len(p) > 0]) if any(len(p) > 0 for p in all_pts) else None
    midline_method = config.get("midline_method", "median")
    if combined is not None and len(combined) > 0:
        mid_cols[side] = float((combined[:, 1].min() + combined[:, 1].max()) / 2) if midline_method == "range_center" else float(np.median(combined[:, 1]))
    else:
        mid_cols[side] = mask_2d.shape[1] / 2
# 画两条中线（左右膝内外侧分割线）
for side in ["left", "right"]:
    ax.axvline(x=mid_cols[side], color="magenta", linestyle="--", lw=1.5, alpha=0.9, label="midline" if side == "left" else None)

# 画边缘点：内侧用冷色(cyan/blue)，外侧用暖色(orange/red)，便于区分内外
if len(med_fem_all) > 0:
    ax.scatter(med_fem_all[:, 1], med_fem_all[:, 0], c="cyan", s=8, edgecolors="darkcyan", linewidths=0.3, label="medial femur")
if len(med_tib_all) > 0:
    ax.scatter(med_tib_all[:, 1], med_tib_all[:, 0], c="blue", s=8, edgecolors="navy", linewidths=0.3, label="medial tibia")
if len(lat_fem_all) > 0:
    ax.scatter(lat_fem_all[:, 1], lat_fem_all[:, 0], c="orange", s=8, edgecolors="darkorange", linewidths=0.3, label="lateral femur")
if len(lat_tib_all) > 0:
    ax.scatter(lat_tib_all[:, 1], lat_tib_all[:, 0], c="red", s=8, edgecolors="darkred", linewidths=0.3, label="lateral tibia")

ax.legend(loc="upper right", fontsize=7)
ax.set_title("Step 3: Medial/Lateral Split (image+mask+edges+midlines)\nmedial=cyan/blue, lateral=orange/red")
ax.axis("off")
plt.tight_layout()
plt.show()

## Step 4: 计算关节间隙宽度（JSN）

对 4 个区室（左/右 × 内/外）计算 JSN，并可视化最小距离连线。

- **第一张图**：与 Step 3 一致底图 + 骨缘散点 + 四区室最短距离（虚线），便于对照算法细节。
- **第二张图（展示用，参考临床排版）**：**纯 X 光灰度**（不叠 mask）、**不画中线**；各区室在股骨–胫骨最近点对之间画 **若干条等距竖线**（默认品红）；**裁剪放大**至四区室包络，便于看间隙。无大端点标记。

In [ ]:
# Compute JSN for all 4 compartments via jsn module
jsn_result = jsn.aggregate_jsn_results(edges_result, spacing, config)

print("\n=== JSN（4 区室）===")
for side in ["left", "right"]:
    for part in ["medial", "lateral"]:
        key_mm = f"jsn_{side}_{part}_mm"
        key_status = f"jsn_{side}_{part}_status"
        key_narrow = f"jsn_{side}_{part}_narrow"
        mm = jsn_result.get(key_mm, float("nan"))
        status = jsn_result.get(key_status, "")
        narrow = jsn_result.get(key_narrow, False)
        narrow_str = " [NARROW]" if narrow else ""
        print(f"  {side} {part}: {mm:.2f} mm (status={status}){narrow_str}")

In [ ]:
# Step 4: 在 Step 3 同一视图上标出各区室 JSN（mm）与是否狭窄
overlay_step4 = make_image_mask_overlay(image_2d, mask_2d, label_mapping, alpha=0.5)
fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.imshow(overlay_step4)

med_fem_all = np.zeros((0, 2))
med_tib_all = np.zeros((0, 2))
lat_fem_all = np.zeros((0, 2))
lat_tib_all = np.zeros((0, 2))
mid_cols = {}
for side in ["left", "right"]:
    if side not in edges_result:
        mid_cols[side] = mask_2d.shape[1] / 2
        continue
    fem_med, tib_med = edges_result[side]["medial"]
    fem_lat, tib_lat = edges_result[side]["lateral"]
    if len(med_fem_all) > 0 and len(fem_med) > 0:
        med_fem_all = np.vstack([med_fem_all, fem_med])
    elif len(fem_med) > 0:
        med_fem_all = fem_med.copy()
    if len(med_tib_all) > 0 and len(tib_med) > 0:
        med_tib_all = np.vstack([med_tib_all, tib_med])
    elif len(tib_med) > 0:
        med_tib_all = tib_med.copy()
    if len(lat_fem_all) > 0 and len(fem_lat) > 0:
        lat_fem_all = np.vstack([lat_fem_all, fem_lat])
    elif len(fem_lat) > 0:
        lat_fem_all = fem_lat.copy()
    if len(lat_tib_all) > 0 and len(tib_lat) > 0:
        lat_tib_all = np.vstack([lat_tib_all, tib_lat])
    elif len(tib_lat) > 0:
        lat_tib_all = tib_lat.copy()
    all_pts = [fem_med, tib_med, fem_lat, tib_lat]
    combined = np.vstack([p for p in all_pts if len(p) > 0]) if any(len(p) > 0 for p in all_pts) else None
    midline_method = config.get("midline_method", "median")
    if combined is not None and len(combined) > 0:
        mid_cols[side] = float((combined[:, 1].min() + combined[:, 1].max()) / 2) if midline_method == "range_center" else float(np.median(combined[:, 1]))
    else:
        mid_cols[side] = mask_2d.shape[1] / 2


for side in ["left", "right"]:
    ax.axvline(x=mid_cols[side], color="magenta", linestyle="--", lw=1.5, alpha=0.9)
# Smaller points (s=4) so compartments stay visible
if len(med_fem_all) > 0:
    ax.scatter(med_fem_all[:, 1], med_fem_all[:, 0], c="cyan", s=4, edgecolors="none", alpha=0.9, label="medial femur")
if len(med_tib_all) > 0:
    ax.scatter(med_tib_all[:, 1], med_tib_all[:, 0], c="blue", s=4, edgecolors="none", alpha=0.9, label="medial tibia")
if len(lat_fem_all) > 0:
    ax.scatter(lat_fem_all[:, 1], lat_fem_all[:, 0], c="orange", s=4, edgecolors="none", alpha=0.9, label="lateral femur")
if len(lat_tib_all) > 0:
    ax.scatter(lat_tib_all[:, 1], lat_tib_all[:, 0], c="red", s=4, edgecolors="none", alpha=0.9, label="lateral tibia")
ax.legend(loc="upper right", fontsize=7)

# Distinct color per compartment for min-distance line + arrows + labels (English)
COMPARTMENT_COLORS = {"left_medial": "cyan", "left_lateral": "orange", "right_medial": "lime", "right_lateral": "magenta"}
COMPARTMENT_LABELS = {"left_medial": "L-med", "left_lateral": "L-lat", "right_medial": "R-med", "right_lateral": "R-lat"}
for side in ["left", "right"]:
    for part in ["medial", "lateral"]:
        key = f"{side}_{part}"
        f_pt = jsn_result.get(f"jsn_{side}_{part}_femur_pt")
        t_pt = jsn_result.get(f"jsn_{side}_{part}_tibia_pt")
        jsn_mm = jsn_result.get(f"jsn_{side}_{part}_mm", float("nan"))
        is_narrow = jsn_result.get(f"jsn_{side}_{part}_narrow", False)
        if f_pt is not None and t_pt is not None and not np.isnan(jsn_mm):
            draw_min_distance_pair(ax, f_pt, t_pt, jsn_mm, overlay_step4.shape,
                                  compartment_label=COMPARTMENT_LABELS[key], color=COMPARTMENT_COLORS[key], draw_labels=False,
                                  is_narrow=is_narrow)

# English summary (caption below figure; legend outside right — no overlap on image)
lines = []
for side in ["left", "right"]:
    for part in ["medial", "lateral"]:
        key = f"{side}_{part}"
        jsn_mm = jsn_result.get(f"jsn_{side}_{part}_mm", float("nan"))
        is_narrow = jsn_result.get(f"jsn_{side}_{part}_narrow", False)
        lab = COMPARTMENT_LABELS[key]
        if np.isnan(jsn_mm):
            lines.append(f"{lab}: n/a")
        else:
            lines.append(f"{lab}: {jsn_mm:.2f} mm" + (" (narrow)" if is_narrow else " (ok)"))
textstr = "\n".join(lines)

ax.set_title("Step 4a: Technical — edges + dashed min-distance (mm)")
ax.axis("off")
ax.legend(bbox_to_anchor=(1.02, 1.0), loc="upper left", fontsize=7, borderaxespad=0)
plt.tight_layout()
fig.subplots_adjust(right=0.72, bottom=0.24)
fig.text(
    0.5,
    0.02,
    textstr,
    transform=fig.transFigure,
    ha="center",
    va="top",
    fontsize=9,
    family="monospace",
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.95),
)
plt.show()

# --- Step 4b 展示图：纯灰度 X 光、无 mask、无中线；品红竖条纹 + 关节局部放大 ---
presentation_gray = grayscale_display_rgb(image_2d)
fig_p, ax_p = plt.subplots(1, 1, figsize=PRESENTATION_FIGSIZE)
ax_p.imshow(presentation_gray)
JSN_PRESENTATION_COLOR = "magenta"
for side in ["left", "right"]:
    for part in ["medial", "lateral"]:
        key = f"{side}_{part}"
        f_pt = jsn_result.get(f"jsn_{side}_{part}_femur_pt")
        t_pt = jsn_result.get(f"jsn_{side}_{part}_tibia_pt")
        jsn_mm = jsn_result.get(f"jsn_{side}_{part}_mm", float("nan"))
        is_narrow = jsn_result.get(f"jsn_{side}_{part}_narrow", False)
        if f_pt is not None and t_pt is not None and not np.isnan(jsn_mm):
            draw_min_distance_pair(
                ax_p, f_pt, t_pt, jsn_mm, presentation_gray.shape,
                compartment_label=COMPARTMENT_LABELS[key], color=JSN_PRESENTATION_COLOR, draw_labels=False,
                presentation=True, is_narrow=is_narrow, presentation_n_stripes=4,
            )
ax_p.set_title("Step 4b: Presentation — grayscale + magenta stripes (knee crop)")
ax_p.axis("off")
plt.tight_layout()
apply_axis_zoom(ax_p, zoom_limits_from_jsn(jsn_result, image_2d.shape))
fig_p.subplots_adjust(right=0.72, bottom=0.12)
ax_p.legend(
    bbox_to_anchor=(1.02, 0.5),
    loc="center left",
    fontsize=9,
    borderaxespad=0,
)
plt.show()

# Summary table
df_jsn = pd.DataFrame([
    {"side": "left", "part": "medial", "JSN_mm": jsn_result.get("jsn_left_medial_mm"), "narrow": jsn_result.get("jsn_left_medial_narrow"), "status": jsn_result.get("jsn_left_medial_status")},
    {"side": "left", "part": "lateral", "JSN_mm": jsn_result.get("jsn_left_lateral_mm"), "narrow": jsn_result.get("jsn_left_lateral_narrow"), "status": jsn_result.get("jsn_left_lateral_status")},
    {"side": "right", "part": "medial", "JSN_mm": jsn_result.get("jsn_right_medial_mm"), "narrow": jsn_result.get("jsn_right_medial_narrow"), "status": jsn_result.get("jsn_right_medial_status")},
    {"side": "right", "part": "lateral", "JSN_mm": jsn_result.get("jsn_right_lateral_mm"), "narrow": jsn_result.get("jsn_right_lateral_narrow"), "status": jsn_result.get("jsn_right_lateral_status")},
])
display(df_jsn)

## Step 5: 批量测试

对多例运行流水线，并在表格中汇总结果。

In [ ]:
def process_single_case(case_id, config, image_dir, mask_dir):
    """
    Process a single case and return JSN results.
    Returns: dict with case_id and 4 JSN values, or None if failed.
    """
    try:
        image_path = image_dir / f"{case_id}_0000.nrrd"
        mask_path = mask_dir / f"{case_id}.nrrd"
        
        if not mask_path.exists():
            return None
        
        mask_sitk = load_sitk_image(mask_path)
        mask_arr = sitk.GetArrayFromImage(mask_sitk)
        mask_2d = mask_arr[0] if mask_arr.ndim == 3 else mask_arr
        spacing = mask_sitk.GetSpacing()
        label_mapping = config["label_mapping"]
        
        # Infer axis
        axis_info = direction.infer_axis(config, mask_2d, spacing, mask_sitk_or_path=mask_sitk)
        
        # Extract edges (per-side)
        edges_per_side = edges.extract_femur_tibia_edges(mask_2d, spacing, label_mapping, axis_info, config)
        edges_result = compartments.split_medial_lateral(
            edges_per_side,
            mask_2d.shape[1],
            spacing=tuple(spacing) if spacing else None,
            exclude_near_midline_ratio=config.get("exclude_near_midline_ratio"),
            exclude_near_midline_mm=config.get("exclude_near_midline_mm"),
        )
        
        # Compute JSN
        jsn_result = jsn.aggregate_jsn_results(edges_result, spacing, config)
        
        return {
            "case_id": case_id,
            "left_medial_mm": jsn_result.get("jsn_left_medial_mm", float("nan")),
            "left_lateral_mm": jsn_result.get("jsn_left_lateral_mm", float("nan")),
            "right_medial_mm": jsn_result.get("jsn_right_medial_mm", float("nan")),
            "right_lateral_mm": jsn_result.get("jsn_right_lateral_mm", float("nan")),
            "left_medial_narrow": jsn_result.get("jsn_left_medial_narrow", False),
            "left_lateral_narrow": jsn_result.get("jsn_left_lateral_narrow", False),
            "right_medial_narrow": jsn_result.get("jsn_right_medial_narrow", False),
            "right_lateral_narrow": jsn_result.get("jsn_right_lateral_narrow", False),
        }
    except Exception as e:
        print(f"处理 {case_id} 出错: {e}")
        return None

In [ ]:
# Process multiple cases (use all loaded cases, typically 96)
test_case_ids = case_ids  # 全部加载的 case，共 96 个
print(f"正在处理 {len(test_case_ids)} 例...")
results = []
for i, cid in enumerate(test_case_ids):
    r = process_single_case(cid, config, image_dir, mask_dir)
    if r:
        results.append(r)
    print(f"  [{i+1}/{len(test_case_ids)}] {cid}: {'OK' if r else '失败'}")

print(f"\n成功处理 {len(results)} / {len(test_case_ids)} 例")

In [ ]:
# Display results table; optional CSV write (paths from **Notebook 参数**)
JSN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSN_CSV_PATH = JSN_OUTPUT_DIR / JSN_RESULTS_CSV_NAME

if results:
    df = pd.DataFrame(results)
    
    # Format narrow columns for display
    for col in df.columns:
        if col.endswith("_narrow"):
            df[col] = df[col].apply(lambda x: "YES" if x else "")
    
    display(df)
    
    df_save = pd.DataFrame(results)
    if SAVE_JSN_RESULTS_CSV:
        df_save.to_csv(JSN_CSV_PATH, index=False, encoding="utf-8")
        print(f"\n已写入 CSV: {JSN_CSV_PATH}（{len(df_save)} 行）")
    else:
        print(f"\nSAVE_JSN_RESULTS_CSV=False，未写入。目标路径: {JSN_CSV_PATH}（{len(df_save)} 行）")
    
    # Summary statistics
    mm_cols = [c for c in df.columns if c.endswith("_mm")]
    print("\n=== 汇总统计 (mm) ===")
    for col in mm_cols:
        vals = pd.to_numeric(df[col], errors='coerce')
        print(f"  {col}: mean={vals.mean():.2f}, std={vals.std():.2f}, min={vals.min():.2f}, max={vals.max():.2f}")
else:
    print("无结果可展示。")

In [ ]:
# 批量导出 **展示图**（与 Step 4b 一致）：纯灰度 X 光、无 mask、无中线；品红竖条纹 + 关节裁剪。
# 每张保存 **一种** 格式（``FIG_EXT`` / ``FIG_DPI``，与 ``osteophyte.ipynb`` 一致），并 ``plt.show()``。
if BATCH_EXPORT_FIGURES and results and len(results) > 0:
    save_dir = JSN_OUTPUT_FIGURE_DIR
    save_dir.mkdir(parents=True, exist_ok=True)
    n_cases = min(len(results), int(BATCH_MAX_CASES))
    
    for i, r in enumerate(results[:n_cases]):
        cid = r["case_id"]
        fig, ax = plt.subplots(1, 1, figsize=BATCH_FIGSIZE)
        
        image_path = image_dir / f"{cid}_0000.nrrd"
        mask_path = mask_dir / f"{cid}.nrrd"
        mask_sitk = load_sitk_image(mask_path)
        mask_arr = sitk.GetArrayFromImage(mask_sitk)
        mask_2d = mask_arr[0] if mask_arr.ndim == 3 else mask_arr
        spacing = mask_sitk.GetSpacing()
        if image_path.exists():
            image_sitk = load_sitk_image(image_path)
            image_arr = sitk.GetArrayFromImage(image_sitk)
            image_2d = image_arr[0] if image_arr.ndim == 3 else image_arr
        else:
            image_2d = np.zeros_like(mask_2d)
        
        # Same pipeline as Step 4: edges + compartments for drawing
        axis_info = direction.infer_axis(config, mask_2d, spacing, mask_sitk_or_path=mask_sitk)
        edges_per_side = edges.extract_femur_tibia_edges(mask_2d, spacing, label_mapping, axis_info, config)
        edges_result = compartments.split_medial_lateral(
            edges_per_side,
            mask_2d.shape[1],
            spacing=tuple(spacing) if spacing else None,
            exclude_near_midline_ratio=config.get("exclude_near_midline_ratio"),
            exclude_near_midline_mm=config.get("exclude_near_midline_mm"),
        )
        
        jsn_result = jsn.aggregate_jsn_results(edges_result, spacing, config)
        presentation_gray = grayscale_display_rgb(image_2d)
        ax.imshow(presentation_gray)
        JSN_PRESENTATION_COLOR = "magenta"
        COMPARTMENT_LABELS = {"left_medial": "L-med", "left_lateral": "L-lat", "right_medial": "R-med", "right_lateral": "R-lat"}
        for side in ["left", "right"]:
            for part in ["medial", "lateral"]:
                key = f"{side}_{part}"
                f_pt = jsn_result.get(f"jsn_{side}_{part}_femur_pt")
                t_pt = jsn_result.get(f"jsn_{side}_{part}_tibia_pt")
                jsn_mm_val = jsn_result.get(f"jsn_{side}_{part}_mm", float("nan"))
                is_narrow = jsn_result.get(f"jsn_{side}_{part}_narrow", False)
                if f_pt is not None and t_pt is not None and not np.isnan(jsn_mm_val):
                    draw_min_distance_pair(
                        ax, f_pt, t_pt, jsn_mm_val, presentation_gray.shape,
                        compartment_label=COMPARTMENT_LABELS[key], color=JSN_PRESENTATION_COLOR, draw_labels=False,
                        presentation=True, is_narrow=is_narrow, presentation_n_stripes=4,
                    )
        ax.set_title(f"{cid} — presentation (grayscale + magenta + crop)")
        ax.axis("off")
        plt.tight_layout()
        apply_axis_zoom(ax, zoom_limits_from_jsn(jsn_result, image_2d.shape))
        fig.subplots_adjust(right=0.72, bottom=0.12)
        ax.legend(
            bbox_to_anchor=(1.02, 0.5),
            loc="center left",
            fontsize=9,
            borderaxespad=0,
        )
        _jsn_ext = str(FIG_EXT).lstrip(".") or "png"
        _fig_out = save_dir / f"{cid}_jsn.{_jsn_ext}"
        fig.savefig(_fig_out, dpi=FIG_DPI, bbox_inches="tight")
        plt.show()
        plt.close(fig)
elif not BATCH_EXPORT_FIGURES:
    print("BATCH_EXPORT_FIGURES=False，跳过批量出图。")
else:
    print("无结果可可视化。")



## 小结

本 Notebook 演示了完整的 JSN 测量流水线：

1. **Step 1**：加载 X 光图像与分割 mask
2. **Step 2**：用 3 种方法提取股骨下缘与胫骨上缘
3. **Step 3**：内外侧划分可视化（分割线 + 点）
4. **Step 4**：计算 4 区室 JSN；**4a** 技术图（mask 叠图+骨缘点+虚线）；**4b** 展示图（纯灰度、品红竖条纹、无中线、关节裁剪）
5. **Step 5**：批量导出与 4b 一致（`{case_id}_jsn.{FIG_EXT}`）；路径、`FIG_EXT`、`FIG_DPI` 见 **Notebook 参数**（与 **osteophyte.ipynb** 相同参数名）。

主要配置参数：
- `edge_method`：骨缘提取方法（distance_percentile、axis_projection、morphological）
- `distance_method`：JSN 计算方法（min、percentile）
- `jsn_narrow_mm`：关节间隙狭窄判定阈值

## 算法预测与标签对比（多标签分类评估）

将 JSN 过窄预测与人工标签对比，作为多标签分类问题（4 个区室：左内、左外、右内、右外），给出测评指标。

In [ ]:
# 加载带标签结果（优先 CSV，否则 Excel）；目录与文件名见 **Notebook 参数**
JSN_LABEL_DIR = JSN_OUTPUT_DIR
csv_path = JSN_LABEL_DIR / EVAL_CSV_NAME
xlsx_path = JSN_LABEL_DIR / EVAL_XLSX_NAME
if csv_path.exists():
    df_eval = pd.read_csv(csv_path, encoding="utf-8")
    print(f"从 CSV 加载 {len(df_eval)} 行: {csv_path}")
elif xlsx_path.exists():
    df_eval = pd.read_excel(xlsx_path)
    print(f"从 Excel 加载 {len(df_eval)} 行: {xlsx_path}")
else:
    raise FileNotFoundError(f"No label file found in {JSN_LABEL_DIR} (tried jwd_result_w_label.csv, jsn_results_w_labels.xlsx)")

IGNORE_CASE_IDS = EVAL_IGNORE_CASE_IDS
case_id_col = "case_id"
if case_id_col in df_eval.columns:
    df_eval = df_eval[~df_eval[case_id_col].isin(IGNORE_CASE_IDS)].copy()
    print(f"排除 {len(IGNORE_CASE_IDS)} 个 ID；剩余 {len(df_eval)} 行")

# 预测列与标签列
compartments = ["left_medial", "left_lateral", "right_medial", "right_lateral"]
pred_cols = [f"{c}_narrow" for c in compartments]
label_cols = [f"{c}_narrow_label" for c in compartments]
for pc, lc in zip(pred_cols, label_cols):
    if pc not in df_eval.columns or lc not in df_eval.columns:
        raise KeyError(f"Missing columns: need {pred_cols} and {label_cols}")

# 标签规则：没写 = false，1 = true；4 个都没写表示四个区室都未狭窄（全 0）
y_pred = df_eval[pred_cols].astype(int)
y_true = df_eval[label_cols].copy()
y_true = (y_true == 1).astype(int)  # 仅 1 为 true，NaN/其他均为 0
y_true = y_true.fillna(0).astype(int)

y_true_v = y_true
y_pred_v = y_pred
n_valid = len(df_eval)
print(f"评估 {n_valid} 例（标签 1=狭窄，否则=未狭窄；全空=全部未狭窄）")

# ---------- 多标签测评指标 ----------
def multilabel_metrics(y_true, y_pred, labels):
    """Per-label 与 overall 指标。y_true, y_pred: (n, 4) 0/1."""
    from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, hamming_loss
    out = {}
    for i, name in enumerate(labels):
        out[name] = {
            "precision": precision_score(y_true.iloc[:, i], y_pred.iloc[:, i], zero_division=0),
            "recall": recall_score(y_true.iloc[:, i], y_pred.iloc[:, i], zero_division=0),
            "f1": f1_score(y_true.iloc[:, i], y_pred.iloc[:, i], zero_division=0),
            "accuracy": accuracy_score(y_true.iloc[:, i], y_pred.iloc[:, i]),
        }
    # macro
    out["macro"] = {
        "precision": np.mean([out[n]["precision"] for n in labels]),
        "recall": np.mean([out[n]["recall"] for n in labels]),
        "f1": np.mean([out[n]["f1"] for n in labels]),
    }
    # micro（按样本-标签对）
    y_t = y_true.values.ravel()
    y_p = y_pred.values.ravel()
    out["micro"] = {
        "precision": precision_score(y_t, y_p, zero_division=0),
        "recall": recall_score(y_t, y_p, zero_division=0),
        "f1": f1_score(y_t, y_p, zero_division=0),
    }
    out["hamming_loss"] = hamming_loss(y_true, y_pred)
    # 按位置比较，避免 y_true/y_pred 列名不同导致 DataFrame 比较报错
    out["subset_accuracy"] = (np.asarray(y_true) == np.asarray(y_pred)).all(axis=1).mean()
    return out

metrics = multilabel_metrics(y_true_v, y_pred_v, compartments)

print("\n=== Per-compartment (多标签各维度) ===")
for name in compartments:
    m = metrics[name]
    print(f"  {name}: P={m['precision']:.4f}, R={m['recall']:.4f}, F1={m['f1']:.4f}, Acc={m['accuracy']:.4f}")

print("\n=== Macro / Micro（多标签整体）===")
print(f"  Macro - P={metrics['macro']['precision']:.4f}, R={metrics['macro']['recall']:.4f}, F1={metrics['macro']['f1']:.4f}")
print(f"  Micro - P={metrics['micro']['precision']:.4f}, R={metrics['micro']['recall']:.4f}, F1={metrics['micro']['f1']:.4f}")
print(f"  汉明损失: {metrics['hamming_loss']:.4f}")
print(f"  子集准确率（完全匹配）: {metrics['subset_accuracy']:.4f}")

# 汇总表
df_metrics = pd.DataFrame([
    {**{"compartment": c}, **metrics[c]} for c in compartments
])
df_metrics = pd.concat([
    df_metrics,
    pd.DataFrame([{"compartment": "macro", **metrics["macro"]}]),
    pd.DataFrame([{"compartment": "micro", **metrics["micro"]}]),
], ignore_index=True)
display(df_metrics)

JSN_LABEL_DIR.mkdir(parents=True, exist_ok=True)
eval_csv_path = JSN_LABEL_DIR / EVAL_METRICS_CSV_NAME
df_metrics.to_csv(eval_csv_path, index=False, encoding="utf-8")
print(f"\n测评指标已保存到 {eval_csv_path}")